# 02 — Attention, from a toy example up to multi-head

This is the heart of the transformer and the thing you'll be asked to whiteboard. So we build it
**in stages**, verifying each one, instead of dropping the final formula on you:

1. The problem attention solves, and the dumbest solution (averaging).
2. A hand-worked **numeric toy example** you can check with a calculator.
3. Single-head self-attention: Q, K, V; scaled dot-product; the causal mask; softmax.
4. Why we divide by √dₖ — demonstrated, not asserted.
5. Multi-head: what breaking into heads buys us.
6. The final `nn.Module` (identical to [`model.py`](../model.py)), checked against PyTorch's own
   fused attention to prove our hand-rolled version is correct.

Everything is tiny and CPU-only. The goal is that by the end you could re-derive this on a
whiteboard from memory.

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1337)

## 1. The problem: tokens need to look at each other

After tokenizing and embedding, a sentence is a stack of vectors, one per token — shape
`(T, C)` for T tokens of C features each. But each vector so far only knows about **its own**
token. To predict the word after *"the animal didn't cross the street because **it** was too
tired"*, the model has to let *"it"* gather information from *"animal"*. Tokens must **communicate**.

The simplest possible communication: each token becomes the **average of itself and all
previous tokens**. ("Previous only" because this is a language model predicting left-to-right —
token *i* must not peek at token *i+1*, which it won't have at generation time. More on this
causal constraint below.)

In [2]:
# toy: 3 tokens, 2 features each
x = torch.tensor([[1., 0.],
                  [0., 2.],
                  [4., 4.]])
T = x.shape[0]

# "average with all previous positions" the slow, obvious way:
avg = torch.zeros_like(x)
for i in range(T):
    avg[i] = x[:i+1].mean(dim=0)     # mean of rows 0..i
print("running average (obvious loop):")
print(avg)

running average (obvious loop):
tensor([[1.0000, 0.0000],
        [0.5000, 1.0000],
        [1.6667, 2.0000]])


Averaging is uniform — every previous token counts equally. That's the weakness: when processing
*"it"*, we don't want *"animal"* and *"the"* weighted the same. We want **data-dependent,
non-uniform weights**. That is *exactly* what attention is: a weighted average where the weights
are computed from the tokens themselves.

First, a key reframe — the same average as a **matrix multiply**. Build a lower-triangular matrix
of weights `W` (row *i* has `1/(i+1)` in its first `i+1` slots, zeros after), then `W @ x`:

In [3]:
W = torch.tril(torch.ones(T, T))
W = W / W.sum(dim=1, keepdim=True)    # normalize each row to sum to 1
print("weight matrix W (row i = how token i mixes tokens 0..i):")
print(W)
print("\nW @ x  == the loop above:")
print(W @ x)
print("match:", torch.allclose(W @ x, avg))

weight matrix W (row i = how token i mixes tokens 0..i):
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])

W @ x  == the loop above:
tensor([[1.0000, 0.0000],
        [0.5000, 1.0000],
        [1.6667, 2.0000]])
match: True


**This is the whole trick.** Attention keeps this structure — `output = weights @ values` — but
replaces the fixed uniform `W` with weights **computed from the data**, and the zeros in the
upper triangle (the causal mask) stay. The rest of this notebook is just: *where do the weights
come from?*

## 2. Q, K, V — a numeric toy example you can check by hand

Every token produces three vectors via three learned linear projections:

- **Query (q)** — "what am I looking for?"
- **Key (k)** — "what do I contain / what can I offer?"
- **Value (v)** — "if you attend to me, here's the information I hand over."

Token *i* attends to token *j* by how well *i*'s query matches *j*'s key: the **dot product**
`qᵢ · kⱼ`. Big dot product ⇒ big raw score ("this key is relevant to my query"). Softmax those
scores into weights; the output is the weighted sum of values.

Let's do it with hand-picked numbers so you can verify every step. We'll use `head_dim = 2` and
skip the learned projections for now (treat x itself as q, k, v) so the arithmetic is transparent.

In [4]:
q = torch.tensor([[1., 0.],   # token 0's query
                  [0., 1.],   # token 1's query
                  [1., 1.]])  # token 2's query
k = q.clone()                 # keys = queries here, just for the demo
v = torch.tensor([[10.,  0.],
                  [ 0., 10.],
                  [ 5.,  5.]])

# raw scores: every query dotted with every key -> (T, T) matrix
scores = q @ k.T
print("raw scores q @ k.T  (row i = token i's scores against all keys):")
print(scores)

raw scores q @ k.T  (row i = token i's scores against all keys):
tensor([[1., 0., 1.],
        [0., 1., 1.],
        [1., 1., 2.]])


Check row 2 by hand: token 2's query is `[1,1]`.
- vs key 0 `[1,0]`: 1·1 + 1·0 = **1**
- vs key 1 `[0,1]`: 1·0 + 1·1 = **1**
- vs key 2 `[1,1]`: 1·1 + 1·1 = **2**

So row 2 is `[1, 1, 2]` — token 2 finds itself most relevant. Matches the printout.

Now the two remaining operations: **scale** by √dₖ (here √2), then **causally mask** future
positions to −∞ so softmax gives them zero weight.

In [5]:
head_dim = q.shape[1]
scaled = scores / math.sqrt(head_dim)              # divide by sqrt(d_k)=sqrt(2)

mask = torch.tril(torch.ones(T, T))                # 1 on/below diagonal, 0 above
scaled_masked = scaled.masked_fill(mask == 0, float("-inf"))
print("after scaling and causal masking (-inf = not allowed to attend):")
print(scaled_masked)

after scaling and causal masking (-inf = not allowed to attend):
tensor([[0.7071,   -inf,   -inf],
        [0.0000, 0.7071,   -inf],
        [0.7071, 0.7071, 1.4142]])


In [6]:
weights = F.softmax(scaled_masked, dim=-1)         # each row -> probabilities summing to 1
print("attention weights (each row sums to 1, upper triangle is exactly 0):")
print(weights)
print("row sums:", weights.sum(dim=-1))

attention weights (each row sums to 1, upper triangle is exactly 0):
tensor([[1.0000, 0.0000, 0.0000],
        [0.3302, 0.6698, 0.0000],
        [0.2483, 0.2483, 0.5035]])
row sums: tensor([1., 1., 1.])


Read the weights: token 0 can only see itself (weight 1.0). Token 1 splits between tokens 0 and
1. Token 2 puts the most weight on itself (its score of 2 was highest) and less on 0 and 1 —
exactly the *non-uniform, data-dependent* mixing we wanted. Finally, `weights @ v`:

In [7]:
out = weights @ v
print("attention output = weights @ v:")
print(out)
print("\ntoken 0 output == value 0 exactly:", torch.allclose(out[0], v[0]))
# token 2, by hand: 0.211*[10,0] + 0.211*[0,10] + 0.577*[5,5]
by_hand = 0.2119*v[0] + 0.2119*v[1] + 0.5761*v[2]
print("token 2 output ~", by_hand.tolist(), "(hand-computed, matches row 2 above)")

attention output = weights @ v:
tensor([[10.0000,  0.0000],
        [ 3.3024,  6.6976],
        [ 5.0000,  5.0000]])

token 0 output == value 0 exactly: True
token 2 output ~ [4.999499797821045, 4.999499797821045] (hand-computed, matches row 2 above)


That's a full attention operation, checked by hand. In code it's five lines:

```python
scores  = q @ k.transpose(-2, -1) / math.sqrt(head_dim)
scores  = scores.masked_fill(mask == 0, float("-inf"))
weights = F.softmax(scores, dim=-1)
out     = weights @ v
```

Two things were *given* in the toy (q, k, v and the head_dim). In a real model, q/k/v come from
**learned linear layers** applied to x, so the model learns *what* to look for. Let's add those.

## 3. Single-head self-attention as a module

"**Self**-attention" = q, k, v are all projections of the **same** input x (a token asks, offers,
and delivers, all about itself). The three projections are the only learned parameters here.

In [8]:
class SingleHeadAttention(nn.Module):
    def __init__(self, n_embd, head_dim, block_size):
        super().__init__()
        # three independent linear maps x -> q, k, v. bias=False is conventional
        # (the layernorm before this already handles centering).
        self.query = nn.Linear(n_embd, head_dim, bias=False)
        self.key   = nn.Linear(n_embd, head_dim, bias=False)
        self.value = nn.Linear(n_embd, head_dim, bias=False)
        # causal mask as a buffer: not a parameter, but moves with .to(device)
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.query(x), self.key(x), self.value(x)   # each (B, T, head_dim)
        scores = q @ k.transpose(-2, -1) / math.sqrt(k.shape[-1])   # (B, T, T)
        scores = scores.masked_fill(self.mask[:T, :T] == 0, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        return weights @ v                                    # (B, T, head_dim)

# shape check on a real-ish batch
attn = SingleHeadAttention(n_embd=32, head_dim=16, block_size=8)
x = torch.randn(4, 8, 32)          # batch=4, T=8, C=32
print("input :", tuple(x.shape))
print("output:", tuple(attn(x).shape), "  (B, T, head_dim)")

input : (4, 8, 32)
output: (4, 8, 16)   (B, T, head_dim)


### Verifying the causal mask actually works

The single most important property: **row *i* of the weights must be zero for all columns > i.**
If it isn't, information leaks from the future and the model cheats during training (it would
"predict" the next token by literally attending to it), then collapses at generation time when
the future doesn't exist. Let's prove our mask holds — and prove it *fails* without the mask.

In [9]:
with torch.no_grad():
    q, k, v = attn.query(x), attn.key(x), attn.value(x)
    scores = q @ k.transpose(-2, -1) / math.sqrt(k.shape[-1])
    masked_w   = F.softmax(scores.masked_fill(attn.mask[:8,:8]==0, float('-inf')), dim=-1)
    unmasked_w = F.softmax(scores, dim=-1)

upper = torch.triu(torch.ones(8, 8), diagonal=1).bool()   # strictly-future positions
print("max weight on any FUTURE position, WITH mask   :", masked_w[0][upper].max().item())
print("max weight on any FUTURE position, WITHOUT mask :", unmasked_w[0][upper].max().item())
assert masked_w[0][upper].max().item() == 0.0
print("\ncausal mask verified: zero attention to the future.")

max weight on any FUTURE position, WITH mask   : 0.0
max weight on any FUTURE position, WITHOUT mask : 0.20815230906009674

causal mask verified: zero attention to the future.


There's also a concrete test of *causality*: changing a **later** token must never alter the
output of an **earlier** one. Let's confirm — perturb the last token, check token 0's output is
byte-for-byte identical.

In [10]:
x2 = x.clone()
x2[:, -1, :] += 100.0                 # violently change the LAST token
with torch.no_grad():
    o1, o2 = attn(x), attn(x2)
print("token 0 output unchanged when the LAST token changes:",
      torch.allclose(o1[:, 0], o2[:, 0]))
print("token 7 output DID change (it can see the last token):",
      not torch.allclose(o1[:, 7], o2[:, 7]))

token 0 output unchanged when the LAST token changes: True
token 7 output DID change (it can see the last token): True


## 4. Why divide by √dₖ? (the classic interview question)

The scores are dot products of dₖ-dimensional vectors. If q and k have entries with ~unit
variance, then `q·k = Σ qₐ kₐ` is a sum of dₖ independent ~unit-variance terms, so its variance
is ~**dₖ** and its std is ~**√dₖ**. As dₖ grows, the raw scores get larger in magnitude.

Why does that hurt? **Softmax saturates.** Feed it logits that are far apart and it returns a
near one-hot distribution — one token gets ~all the weight, the rest ~0. And softmax's gradient
is `p·(1−p)`-ish: for probabilities pinned near 0 or 1, the gradient **vanishes**. Training
stalls. Dividing by √dₖ rescales the scores back to ~unit variance regardless of head size,
keeping softmax in its responsive, high-gradient regime. Let's watch it happen.

In [11]:
for d in [8, 64, 512]:
    q = torch.randn(2000, d); k = torch.randn(2000, d)
    raw    = (q * k).sum(-1)
    scaled = raw / math.sqrt(d)
    print(f"d_k={d:4d} | raw scores std = {raw.std():7.2f}  (~sqrt(d)={math.sqrt(d):5.1f})"
          f" | after /sqrt(d) = {scaled.std():.2f}")

d_k=   8 | raw scores std =    2.87  (~sqrt(d)=  2.8) | after /sqrt(d) = 1.02
d_k=  64 | raw scores std =    7.97  (~sqrt(d)=  8.0) | after /sqrt(d) = 1.00
d_k= 512 | raw scores std =   22.52  (~sqrt(d)= 22.6) | after /sqrt(d) = 1.00


In [12]:
# and the downstream effect on softmax sharpness. Take real scores from a
# d=512 head and compare softmax WITH vs WITHOUT the 1/sqrt(d) scaling:
d = 512
q = torch.randn(8, d); k = torch.randn(8, d)
raw = q @ k.T                                  # (8, 8) scores, std ~ sqrt(d)
for label, s in [("unscaled", raw), ("scaled  ", raw / math.sqrt(d))]:
    p = F.softmax(s, dim=-1)
    peak = p.max(dim=-1).values.mean().item()  # how one-hot is the distribution?
    note = "softmax saturated, gradients ~0" if peak > 0.9 else "healthy, informative gradients"
    print(f"{label}: mean peak prob = {peak:.3f}   <- {note}")

unscaled: mean peak prob = 0.933   <- softmax saturated, gradients ~0
scaled  : mean peak prob = 0.365   <- healthy, informative gradients


Without the `1/√dₖ`, a 512-dim head produces scores with std ~23; softmax of those is essentially
`argmax`, and the model can't learn *how much* to attend, only *whether*. The scaling is the one
line that keeps attention differentiable at scale.

## 5. Multi-head attention: several conversations at once

A single head computes **one** set of attention weights — one notion of "relevant." But *"it →
animal"* (coreference), *"tired → was"* (subject-verb), and *"street → the"* (article-noun) are
different relations you'd want tracked simultaneously. One head has to average them together.

**Multi-head** runs `h` attention operations in parallel, each on its own `C/h`-dim slice, then
concatenates the results and mixes them with an output projection. Each head is free to
specialize. Crucially it costs almost nothing: `h` heads of dim `C/h` is the same total width
`C` as one head of dim `C` — we **split** the channels, not add to them.

In [13]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        # ONE fused projection produces q,k,v for ALL heads (3*n_embd wide),
        # then we reshape into heads. Mathematically identical to n_head separate
        # SingleHeadAttention modules; one big matmul is just faster on GPU.
        self.qkv_proj = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.out_proj = nn.Linear(n_embd, n_embd, bias=False)   # mixes heads back together
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size))
                                          .view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv_proj(x).split(C, dim=2)              # each (B, T, C)
        hd = C // self.n_head
        # (B, T, C) -> (B, n_head, T, head_dim): the transpose puts n_head next
        # to batch so the matmul below runs each head independently and in parallel
        q = q.view(B, T, self.n_head, hd).transpose(1, 2)
        k = k.view(B, T, self.n_head, hd).transpose(1, 2)
        v = v.view(B, T, self.n_head, hd).transpose(1, 2)

        scores = q @ k.transpose(-2, -1) / math.sqrt(hd)      # (B, n_head, T, T)
        scores = scores.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        y = weights @ v                                       # (B, n_head, T, head_dim)
        y = y.transpose(1, 2).contiguous().view(B, T, C)      # concat heads back to (B,T,C)
        return self.out_proj(y)

mha = MultiHeadAttention(n_embd=32, n_head=4, block_size=8)
x = torch.randn(4, 8, 32)
print("MHA output shape:", tuple(mha(x).shape), " (same (B,T,C) as input -> stackable)")
print("head_dim per head:", 32 // 4, " x 4 heads =", 32, "= full width, no extra cost")

MHA output shape: (4, 8, 32)  (same (B,T,C) as input -> stackable)
head_dim per head: 8  x 4 heads = 32 = full width, no extra cost


### The reshape, spelled out

The `.view(B, T, n_head, hd).transpose(1, 2)` is where people get lost, so trace the shapes:

| step | shape | meaning |
|---|---|---|
| after `c_attn`, one of q/k/v | `(B, T, C)` | every token has a C-dim vector |
| `.view(B, T, n_head, hd)` | `(B, T, nh, hd)` | split the C channels into nh groups of hd |
| `.transpose(1, 2)` | `(B, nh, T, hd)` | put heads beside batch → each head is an independent `(T, hd)` attention problem |
| `weights @ v` | `(B, nh, T, hd)` | attention done per head, in one batched matmul |
| `.transpose(1,2).view(B,T,C)` | `(B, T, C)` | glue the heads' outputs back side by side |

No Python loop over heads — the `nh` dimension rides along inside the batched matmul, so all
heads compute in parallel. This is why the fused `c_attn` + reshape pattern is universal.

## 6. Correctness check against PyTorch's own attention

We claim our five lines implement the same function as PyTorch's fused
`F.scaled_dot_product_attention` (the FlashAttention kernel used in production). If so, feeding
identical q/k/v with `is_causal=True` should give identical outputs. This both validates our code
*and* shows what production swaps in for speed.

In [14]:
B, nh, T, hd = 2, 4, 16, 8
q, k, v = torch.randn(3, B, nh, T, hd)

# ours, by hand
scores = q @ k.transpose(-2, -1) / math.sqrt(hd)
cmask  = torch.tril(torch.ones(T, T)).view(1, 1, T, T)
ours   = F.softmax(scores.masked_fill(cmask == 0, float("-inf")), dim=-1) @ v

# PyTorch's fused kernel (same math, optimized; used in real training)
theirs = F.scaled_dot_product_attention(q, k, v, is_causal=True)

print("max abs difference vs torch's fused attention:", (ours - theirs).abs().max().item())
assert torch.allclose(ours, theirs, atol=1e-6)
print("identical (to fp32 rounding). Our hand-rolled attention is correct.")

max abs difference vs torch's fused attention: 3.5762786865234375e-07
identical (to fp32 rounding). Our hand-rolled attention is correct.


## Takeaways

- Attention = **`output = softmax(mask(QKᵀ/√dₖ)) · V`** — a *data-dependent weighted average* of
  value vectors, where the weights come from query·key similarity.
- **Q/K/V** are three learned projections of the input: query = what I seek, key = what I offer,
  value = what I pass along.
- The **causal mask** (−∞ on future positions before softmax) is what makes it a *left-to-right*
  language model; we verified no weight ever lands on the future and that earlier tokens are
  unaffected by later ones.
- **√dₖ scaling** keeps score variance ~1 so softmax stays in its high-gradient regime — without
  it, big heads make softmax collapse to argmax and gradients vanish.
- **Multi-head** = the same channels *split* into parallel heads so different heads track
  different relations, then recombined by an output projection — extra expressivity, ~zero extra
  cost.
- Our from-scratch version matches `F.scaled_dot_product_attention` exactly; production just uses
  the fused kernel for speed and memory.

**The O(T²) cost:** the scores matrix is `(T, T)` per head — attention's compute and memory grow
**quadratically** with context length. This single fact drives most of modern LLM engineering
(FlashAttention, sliding-window/sparse attention, and the KV cache in notebook 07).

**Interview quick-fire:**
- *Why divide by √dₖ?* → dot products of dₖ-dim vectors have std ~√dₖ; unscaled, softmax saturates and gradients vanish.
- *What breaks without the causal mask?* → the model attends to future tokens, "cheats" at training, and produces garbage at inference.
- *What do multiple heads buy you over one big head?* → parallel, specialized attention patterns (syntax, coreference, position) instead of one averaged-together pattern, at the same parameter cost.
- *Where does the O(T²) come from?* → the QKᵀ scores matrix is T×T per head.

**Next:** [03 — The transformer block and full GPT](03_transformer_architecture.ipynb): wrap
attention in residuals + LayerNorm, add the MLP, stack it, and attach embeddings and the LM head.